# 11 — Scenario and change reports

**Audience:** Users documenting IAM inputs and database transformations from a Premise build.

**Prerequisites:** A configured source database and a `NewDatabase` instance that has completed at least one update.

**Learning goals:** generate reports explicitly, locate their files, and inspect report structure before analysis.


## Outline

1. Build a small updated scenario with automatic reports disabled.
2. Generate a named scenario report.
3. Generate the change report from transformation logs.


In [ ]:
import os
from pathlib import Path

import bw2data as bd
import pandas as pd

from premise import NewDatabase

PROJECT = "ecoinvent-3.12-cutoff"
SOURCE_DATABASE = "ecoinvent-3.12-cutoff"
BIOSPHERE_DATABASE = "ecoinvent-3.12-biosphere"
PREMISE_KEY = os.environ.get("PREMISE_KEY")
REPORT_DIR = Path("export/tutorial-reports")

if not PREMISE_KEY:
    raise RuntimeError("Set PREMISE_KEY before running this tutorial.")
bd.projects.set_current(PROJECT)


## 1. Build a focused scenario

`generate_reports=False` prevents export methods from generating reports automatically, which makes the manual calls below easier to see.


In [ ]:
ndb = NewDatabase(
    scenarios=[
        {"model": "remind", "pathway": "SSP2-NDC", "year": 2030},
    ],
    source_db=SOURCE_DATABASE,
    source_version="3.12",
    biosphere_name=BIOSPHERE_DATABASE,
    key=PREMISE_KEY,
    generate_reports=False,
)
ndb.update(["electricity"])


## 2. Generate and inspect the scenario report

The scenario report summarizes IAM variables used by the run.


In [ ]:
SCENARIO_REPORT_NAME = "scenario_report.xlsx"
ndb.generate_scenario_report(
    filepath=REPORT_DIR,
    name=SCENARIO_REPORT_NAME,
)

scenario_report_path = REPORT_DIR / SCENARIO_REPORT_NAME
pd.ExcelFile(scenario_report_path).sheet_names


## 3. Generate the change report

The change report summarizes created and modified datasets, performance indicators, scaling factors, and validation flags. It is written below `export/change reports/`.


In [ ]:
ndb.generate_change_report()
change_report_directory = Path.cwd() / "export" / "change reports"
sorted(change_report_directory.glob("*.xlsx"))[-5:]


## Pitfalls and extension

- Generate the change report after updates, while the relevant logs are available.
- Treat validation sheets as a review queue, not automatic proof that every modeling choice is correct.
- Extension: leave `generate_reports=True` to generate reports automatically during export.

## Exercise

Load the first worksheet of the scenario report and list its columns without printing the whole table.


In [ ]:
exercise_frame = pd.read_excel(scenario_report_path, sheet_name=0)
exercise_frame.columns.tolist()
